# Olist Data Preparation and Order-Level Merge
This notebook prepares nine original Olist CSV files and produces one validated analytical record per order.

It checks source keys and relationships, summarizes child records, joins customer and seller information, and derives fulfillment measurements and preliminary outcome indicators.

Detailed Milestone 2 analysis is performed separately in 02_milestone2_analysis.ipynb.

## 1. Environment and Data Loading

The workflow starts from nine original CSV files. Derived data and analysis outputs are generated by the notebook.

In [4]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

# Support launching Jupyter from either the project root or the notebooks folder.
cwd = Path.cwd()

if (cwd / "data" / "raw").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "raw").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Cannot find data/raw. Place the nine source CSV files "
        "there and launch Jupyter from the project root "
        "or the notebooks folder."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
DERIVED_DIR = PROJECT_ROOT / "data" / "derived"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "figures"

for folder in [DERIVED_DIR, TABLE_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Project root:", PROJECT_ROOT)

Python: 3.13.5
pandas: 2.3.2
NumPy: 2.3.1
Project root: /Users/bishanzhou/Documents/wustl_life/practicum/olist_capstone


In [7]:
SOURCE_FILES = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",}

missing_files = [
    filename
    for filename in SOURCE_FILES.values()
    if not (RAW_DIR / filename).is_file()]

if missing_files:
    raise FileNotFoundError(
        f"Missing source files in {RAW_DIR}: {missing_files}")

zip_dtypes = {
    "customers": {"customer_zip_code_prefix": "string"},
    "sellers": {"seller_zip_code_prefix": "string"},
    "geolocation": {"geolocation_zip_code_prefix": "string"},}

raw_tables = {
    name: pd.read_csv(
        RAW_DIR / filename,
        dtype=zip_dtypes.get(name),)
    for name, filename in SOURCE_FILES.items()}

# Working copies keep the original loaded tables unchanged.
orders = raw_tables["orders"].copy()
customers = raw_tables["customers"].copy()
items = raw_tables["items"].copy()
payments = raw_tables["payments"].copy()
reviews = raw_tables["reviews"].copy()
products = raw_tables["products"].copy()
sellers = raw_tables["sellers"].copy()
geolocation = raw_tables["geolocation"].copy()
category_translation = raw_tables["category_translation"].copy()

In [10]:
source_summary = pd.DataFrame([
    {
        "table": name,
        "source_file": SOURCE_FILES[name],
        "rows": len(df),
        "columns": df.shape[1],
        "exact_duplicate_rows": int(df.duplicated().sum()),}
    for name, df in raw_tables.items()])

display(source_summary)

source_summary.to_csv(
    TABLE_DIR / "source_table_summary.csv",
    index=False,)

display(orders.head())

,table,source_file,rows,columns,exact_duplicate_rows
0,orders,olist_orders_dataset.csv,99441,8,0
1,customers,olist_customers_dataset.csv,99441,5,0
2,items,olist_order_items_dataset.csv,112650,7,0
3,payments,olist_order_payments_dataset.csv,103886,5,0
4,reviews,olist_order_reviews_dataset.csv,99224,7,0
5,products,olist_products_dataset.csv,32951,9,0
6,sellers,olist_sellers_dataset.csv,3095,4,0
7,geolocation,olist_geolocation_dataset.csv,1000163,5,261831
8,category_translation,product_category_name_translation.csv,71,2,0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 2. Keys and Table Relationships

Before merging, we check:
- missing and duplicated join keys;
- whether child records match their parent tables;
- how many item, payment, and review records belong to each order.

Multiple child records are not automatically errors.
They must be handled before constructing one row per order.

In [13]:
KEY_CHECKS = [
    ("orders", ["order_id"]),
    ("customers", ["customer_id"]),
    ("products", ["product_id"]),
    ("sellers", ["seller_id"]),
    ("category_translation", ["product_category_name"]),
    ("items", ["order_id", "order_item_id"]),
    ("payments", ["order_id", "payment_sequential"]),]

key_results = []

for table_name, keys in KEY_CHECKS:
    df = raw_tables[table_name]

    key_results.append({
        "table": table_name,
        "key": " + ".join(keys),
        "rows_with_missing_key": int(
            df[keys].isna().any(axis=1).sum()
        ),
        "extra_rows_with_duplicated_key": int(
            df.duplicated(subset=keys).sum()
        ),})

key_summary = pd.DataFrame(key_results)
display(key_summary)

key_summary.to_csv(
    TABLE_DIR / "key_checks.csv",
    index=False,)

assert key_summary["rows_with_missing_key"].eq(0).all(), (
    "Missing keys found. Inspect them before merging.")

assert key_summary["extra_rows_with_duplicated_key"].eq(0).all(), (
    "Duplicated keys found. Inspect them before merging.")

,table,key,rows_with_missing_key,extra_rows_with_duplicated_key
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,products,product_id,0,0
3,sellers,seller_id,0,0
4,category_translation,product_category_name,0,0
5,items,order_id + order_item_id,0,0
6,payments,order_id + payment_sequential,0,0


In [14]:
RELATIONSHIP_CHECKS = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("items", "order_id", "orders", "order_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("items", "product_id", "products", "product_id"),
    ("items", "seller_id", "sellers", "seller_id"),
    (
        "products",
        "product_category_name",
        "category_translation",
        "product_category_name",
    ),]

relationship_results = []

for child_name, child_key, parent_name, parent_key in RELATIONSHIP_CHECKS:
    child = raw_tables[child_name]
    parent = raw_tables[parent_name]

    # Separate absent keys from observed keys without a match.
    observed = child[child_key].notna()
    matched = child[child_key].isin(parent[parent_key].dropna())
    unmatched = observed & ~matched

    relationship_results.append({
        "relationship": f"{child_name} → {parent_name}",
        "join_key": child_key,
        "child_rows": len(child),
        "missing_child_key_rows": int((~observed).sum()),
        "unmatched_nonmissing_key_rows": int(unmatched.sum()),
        "unmatched_distinct_keys": int(
            child.loc[unmatched, child_key].nunique()
        ),})

relationship_summary = pd.DataFrame(relationship_results)
display(relationship_summary)

relationship_summary.to_csv(
    TABLE_DIR / "relationship_checks.csv",
    index=False,)

,relationship,join_key,child_rows,missing_child_key_rows,unmatched_nonmissing_key_rows,unmatched_distinct_keys
0,orders → customers,customer_id,99441,0,0,0
1,items → orders,order_id,112650,0,0,0
2,payments → orders,order_id,103886,0,0,0
3,reviews → orders,order_id,99224,0,0,0
4,items → products,product_id,112650,0,0,0
5,items → sellers,seller_id,112650,0,0,0
6,products → category_translation,product_category_name,32951,610,13,2


In [16]:
child_record_counts = (
    orders[["order_id"]]
    .set_index("order_id"))
for name, df in [
    ("items", items),
    ("payments", payments),
    ("reviews", reviews),]:
    counts = df.groupby("order_id").size()

    child_record_counts[f"{name}_record_count"] = (
        counts.reindex(child_record_counts.index)
        .fillna(0)
        .astype("int64"))

multiplicity_results = []

for column in child_record_counts.columns:
    counts = child_record_counts[column]

    multiplicity_results.append({
        "child_table": column.replace("_record_count", ""),
        "orders_with_no_records": int(counts.eq(0).sum()),
        "orders_with_one_record": int(counts.eq(1).sum()),
        "orders_with_multiple_records": int(counts.gt(1).sum()),
        "maximum_records_per_order": int(counts.max()),})

multiplicity_summary = pd.DataFrame(multiplicity_results)
display(multiplicity_summary)

multiplicity_summary.to_csv(
    TABLE_DIR / "records_per_order_summary.csv",
    index=False,)

,child_table,orders_with_no_records,orders_with_one_record,orders_with_multiple_records,maximum_records_per_order
0,items,775,88863,9803,21
1,payments,1,96479,2961,29
2,reviews,768,98126,547,3


In [18]:
review_structure = (
    reviews.groupby("order_id")
    .agg(
        review_record_count=("review_id", "size"),
        distinct_scores=("review_score", "nunique"),
        score_min=("review_score", "min"),
        score_max=("review_score", "max"),))

multiple_reviews = review_structure[
    review_structure["review_record_count"].gt(1)].copy()

conflicting_reviews = multiple_reviews[
    multiple_reviews["distinct_scores"].gt(1)].copy()

print("Orders with multiple reviews:", len(multiple_reviews))
print("Orders with conflicting scores:", len(conflicting_reviews))

display(conflicting_reviews.head())

review_structure.to_csv(
    TABLE_DIR / "review_record_structure.csv",
    index=True,)

Orders with multiple reviews: 547
Orders with conflicting scores: 202


,review_record_count,distinct_scores,score_min,score_max
order_id,,,,
013056cfe49763c6f66bda03396c5ee3,2,2,4,5
02355020fd0a40a0d56df9f6ff060413,2,2,1,3
029863af4b968de1e5d6a82782e662f5,2,2,4,5
03c939fd7fd3b38f8485a0f95798f1f6,3,2,3,4
03eba6d9fef8f5b3e811d4b5a7cca9cd,2,2,4,5


## 3. Preparing Order-Level Records

We parse timestamps and summarize item, payment, and review records
before joining them to orders.

Missing measurements remain missing. Multiple reviews are summarized,
and the latest submitted review is provisionally selected as the
primary outcome record.

In [20]:
TIMESTAMP_COLUMNS = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "items": ["shipping_limit_date"],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp",],}

working_tables = {
    "orders": orders,
    "items": items,
    "reviews": reviews,}

timestamp_results = []

for table_name, columns in TIMESTAMP_COLUMNS.items():
    df = working_tables[table_name]

    for column in columns:
        original = df[column].copy()
        parsed = pd.to_datetime(original, errors="coerce")

        timestamp_results.append({
            "table": table_name,
            "column": column,
            "original_missing": int(original.isna().sum()),
            "parse_failures": int(
                (original.notna() & parsed.isna()).sum()
            ),})

        df[column] = parsed

timestamp_summary = pd.DataFrame(timestamp_results)
display(timestamp_summary)

timestamp_summary.to_csv(
    TABLE_DIR / "timestamp_parsing_checks.csv",
    index=False,)

,table,column,original_missing,parse_failures
0,orders,order_purchase_timestamp,0,0
1,orders,order_approved_at,160,0
2,orders,order_delivered_carrier_date,1783,0
3,orders,order_delivered_customer_date,2965,0
4,orders,order_estimated_delivery_date,0,0
5,items,shipping_limit_date,0,0
6,reviews,review_creation_date,0,0
7,reviews,review_answer_timestamp,0,0


In [22]:
products_enriched = products.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one",)

items_enriched = items.merge(
    products_enriched,
    on="product_id",
    how="left",
    validate="many_to_one",)

assert len(items_enriched) == len(items)

items_enriched["product_volume_cm3"] = (
    items_enriched["product_length_cm"]
    * items_enriched["product_height_cm"]
    * items_enriched["product_width_cm"])

display(items_enriched.head())

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,product_volume_cm3
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,3528.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,60000.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,14157.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,2400.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,42000.0


In [24]:
def sorted_unique_values(series):
    """Retain distinct values in a deterministic order."""
    values = sorted(set(series.dropna().astype(str)))
    return "|".join(values) if values else pd.NA

def most_frequent_category(series):
    """Select the modal category; break ties alphabetically."""
    counts = series.dropna().astype(str).value_counts()

    if counts.empty:
        return pd.NA

    tied_categories = counts[counts.eq(counts.max())].index
    return sorted(tied_categories)[0]


def sum_preserving_missing(series):
    """Return missing when all values are missing."""
    return series.sum(min_count=1)

In [27]:
item_summary = (
    items_enriched.groupby("order_id")
    .agg(
        item_count=("order_item_id", "size"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        category_count=("product_category_name", "nunique"),

        items_price_total=("price", sum_preserving_missing),
        freight_value_total=("freight_value", sum_preserving_missing),

        primary_product_category=(
            "product_category_name",
            most_frequent_category,),
        product_categories=(
            "product_category_name",
            sorted_unique_values,),
        product_categories_english=(
            "product_category_name_english",
            sorted_unique_values,),
        product_ids=("product_id", sorted_unique_values),
        seller_ids=("seller_id", sorted_unique_values),

        shipping_limit_earliest=("shipping_limit_date", "min"),
        shipping_limit_latest=("shipping_limit_date", "max"),

        product_weight_g_mean=("product_weight_g", "mean"),
        product_volume_cm3_mean=("product_volume_cm3", "mean"),

        item_missing_category_count=(
            "product_category_name",
            lambda s: int(s.isna().sum()),),
        item_missing_weight_count=(
            "product_weight_g",
            lambda s: int(s.isna().sum()),),
        item_missing_volume_count=(
            "product_volume_cm3",
            lambda s: int(s.isna().sum()),),)
    .reset_index())

In [29]:
# Translate the selected original category, rather than independently selecting a modal English category.
translation_lookup = category_translation.set_index(
    "product_category_name")["product_category_name_english"]

item_summary["primary_product_category_english"] = (
    item_summary["primary_product_category"].map(translation_lookup))

assert item_summary["order_id"].is_unique
assert np.isclose(
    item_summary["items_price_total"].sum(),
    items["price"].sum(),)

assert np.isclose(
    item_summary["freight_value_total"].sum(),
    items["freight_value"].sum(),)

print("Orders with item records:", len(item_summary))
display(item_summary.head())

Orders with item records: 98666


,order_id,item_count,product_count,seller_count,category_count,items_price_total,freight_value_total,primary_product_category,product_categories,product_categories_english,product_ids,seller_ids,shipping_limit_earliest,shipping_limit_latest,product_weight_g_mean,product_volume_cm3_mean,item_missing_category_count,item_missing_weight_count,item_missing_volume_count,primary_product_category_english
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,1,58.90,13.29,cool_stuff,cool_stuff,cool_stuff,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,2017-09-19 09:45:35,650.0,3528.0,0,0,0,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,1,239.90,19.93,pet_shop,pet_shop,pet_shop,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,2017-05-03 11:05:13,30000.0,60000.0,0,0,0,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,1,1,1,199.00,17.87,moveis_decoracao,moveis_decoracao,furniture_decor,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,2018-01-18 14:48:30,3050.0,14157.0,0,0,0,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,1,12.99,12.79,perfumaria,perfumaria,perfumery,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,2018-08-15 10:10:18,200.0,2400.0,0,0,0,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,1,199.90,18.14,ferramentas_jardim,ferramentas_jardim,garden_tools,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,2017-02-13 13:57:51,3750.0,42000.0,0,0,0,garden_tools


In [33]:
payment_summary = (
    payments.groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "size"),
        payment_value_total=(
            "payment_value",
            sum_preserving_missing,),
        payment_types=("payment_type", sorted_unique_values),
        payment_installments_max=("payment_installments", "max"),
        payment_installments_mean=("payment_installments", "mean"),)
    .reset_index())

assert payment_summary["order_id"].is_unique

assert np.isclose(
    payment_summary["payment_value_total"].sum(),
    payments["payment_value"].sum(),)

print("Orders with payment records:", len(payment_summary))
display(payment_summary.head())

Orders with payment records: 99440


,order_id,payment_record_count,payment_value_total,payment_types,payment_installments_max,payment_installments_mean
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,credit_card,2,2.0
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,credit_card,3,3.0
2,000229ec398224ef6ca0657da4fc703e,1,216.87,credit_card,5,5.0
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,credit_card,2,2.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,credit_card,3,3.0


In [35]:
review_summary = (
    reviews.groupby("order_id")
    .agg(
        review_record_count=("review_id", "size"),
        review_score_min=("review_score", "min"),
        review_score_max=("review_score", "max"),
        review_score_mean=("review_score", "mean"),)
    .reset_index())

latest_reviews = (
    reviews.sort_values(
        [
            "order_id",
            "review_answer_timestamp",
            "review_creation_date",
            "review_id",
            "review_score",
        ],
        ascending=True,
        na_position="first",
        kind="stable",)
    .drop_duplicates(subset="order_id", keep="last")
    [[
            "order_id",
            "review_id",
            "review_score",
            "review_creation_date",
            "review_answer_timestamp",
        ]]
    .rename(columns={
        "review_id": "review_id_latest",
        "review_score": "review_score_latest",}))

review_summary = review_summary.merge(
    latest_reviews,
    on="order_id",
    how="left",
    validate="one_to_one",)

In [37]:
# Only assess score disagreement when valid scores exist.
review_summary["review_scores_conflict"] = (
    review_summary["review_score_min"]
    .ne(review_summary["review_score_max"])
    .where(
        review_summary["review_score_min"].notna()
        & review_summary["review_score_max"].notna())
    .astype("boolean"))

assert review_summary["order_id"].is_unique
print("Orders with review records:", len(review_summary))
print(
    "Orders with conflicting scores:",
    int(review_summary["review_scores_conflict"].sum()),)

display(review_summary.head())

Orders with review records: 98673
Orders with conflicting scores: 202


,order_id,review_record_count,review_score_min,review_score_max,review_score_mean,review_id_latest,review_score_latest,review_creation_date,review_answer_timestamp,review_scores_conflict
0,00010242fe8c5a6d1ba2dd792cb16214,1,5,5,5.0,97ca439bc427b48bc1cd7177abe71365,5,2017-09-21,2017-09-22 10:57:03,False
1,00018f77f2f0320c557190d7a144bdd3,1,4,4,4.0,7b07bacd811c4117b742569b04ce3580,4,2017-05-13,2017-05-15 11:34:13,False
2,000229ec398224ef6ca0657da4fc703e,1,5,5,5.0,0c5b33dea94867d1ac402749e5438e8b,5,2018-01-23,2018-01-23 16:06:31,False
3,00024acbcdf0a6daa1e931b038114c75,1,4,4,4.0,f4028d019cb58564807486a6aaf33817,4,2018-08-15,2018-08-15 16:39:01,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5,5,5.0,940144190dcba6351888cafa43f3a3a5,5,2017-03-02,2017-03-03 10:54:59,False


Item and payment records are aggregated before joining to orders, preventing one-to-many joins from multiplying analytical rows.
Their monetary totals reconcile with the original tables.

For reviews, the latest submitted record is a provisional primary outcome choice. Review counts and score ranges are retained for sensitivity analysis. Mean review scores are descriptive summaries, not the primary binary target.

Orders without child records will remain in the final merged table. Their absent measurements will not be interpreted as zero.

## 4. Geographic Preparation and Order-Level Merge

Geographic records are summarized to one representative coordinate per ZIP prefix. Customer and seller locations are then linked to
orders.

All joins retain the original orders. Missing matches remain missing, and row counts are checked to prevent accidental duplication.

In [39]:
valid_coordinates = (
    geolocation["geolocation_lat"].between(-90, 90)
    & geolocation["geolocation_lng"].between(-180, 180))

print("Original geographic rows:", len(geolocation))
print(
    "Exact duplicate geographic rows:",
    int(geolocation.duplicated().sum()),)
print(
    "Rows with missing or globally invalid coordinates:",
    int((~valid_coordinates).sum()),)

geo_unique = (
    geolocation.loc[valid_coordinates]
    .drop_duplicates()
    .copy())

geo_by_zip = (
    geo_unique.groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "median"),
        longitude=("geolocation_lng", "median"),
        geo_unique_row_count=("geolocation_lat", "size"),)
    .reset_index())

assert geo_by_zip["geolocation_zip_code_prefix"].is_unique

print("ZIP prefixes with representative coordinates:", len(geo_by_zip))
display(geo_by_zip.head())

Original geographic rows: 1000163
Exact duplicate geographic rows: 261831
Rows with missing or globally invalid coordinates: 0
ZIP prefixes with representative coordinates: 19015


,geolocation_zip_code_prefix,latitude,longitude,geo_unique_row_count
0,01001,-23.549951,-46.634027,11
1,01002,-23.548228,-46.635247,6
2,01003,-23.548977,-46.635313,11
3,01004,-23.549550,-46.634771,14
4,01005,-23.549763,-46.636100,13


In [41]:
merge_checks = []
def checked_left_merge(left, right, on, name, validate):
    """Preserve left-hand records and document unmatched keys."""
    indicator = "_join_status"

    merged = left.merge(
        right,
        on=on,
        how="left",
        validate=validate,
        indicator=indicator,)

    merge_checks.append({
        "join": name,
        "left_rows": len(left),
        "output_rows": len(merged),
        "unmatched_left_rows": int(
            merged[indicator].eq("left_only").sum()),})

    assert len(merged) == len(left), (
        f"{name}: the join changed the number of rows.")

    return merged.drop(columns=indicator)

In [43]:
customer_geo = geo_by_zip.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "latitude": "customer_latitude",
    "longitude": "customer_longitude",
    "geo_unique_row_count": "customer_geo_unique_row_count",})

customers_enriched = checked_left_merge(
    customers,
    customer_geo,
    on="customer_zip_code_prefix",
    name="Customer ZIP to coordinates",
    validate="many_to_one",)

assert customers_enriched["customer_id"].is_unique

In [48]:
seller_geo = geo_by_zip.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "latitude": "seller_latitude",
    "longitude": "seller_longitude",
    "geo_unique_row_count": "seller_geo_unique_row_count",})

sellers_enriched = checked_left_merge(
    sellers,
    seller_geo,
    on="seller_zip_code_prefix",
    name="Seller ZIP to coordinates",
    validate="many_to_one",)

items_with_sellers = checked_left_merge(
    items_enriched,
    sellers_enriched,
    on="seller_id",
    name="Items to seller information",
    validate="many_to_one",)

seller_location_summary = (
    items_with_sellers.groupby("order_id")
    .agg(
        seller_states=("seller_state", sorted_unique_values),
        seller_cities=("seller_city", sorted_unique_values),
        seller_zip_code_prefixes=(
            "seller_zip_code_prefix",
            sorted_unique_values,),)
    .reset_index())

assert seller_location_summary["order_id"].is_unique

In [47]:
item_locations = checked_left_merge(
    items_with_sellers[
        ["order_id", "seller_latitude", "seller_longitude"]],
    orders[["order_id", "customer_id"]],
    on="order_id",
    name="Item locations to order customer",
    validate="many_to_one",)

item_locations = checked_left_merge(
    item_locations,
    customers_enriched[
        ["customer_id", "customer_latitude", "customer_longitude"]],
    on="customer_id",
    name="Item locations to customer coordinates",
    validate="many_to_one",)

customer_lat = np.radians(item_locations["customer_latitude"])
seller_lat = np.radians(item_locations["seller_latitude"])

longitude_difference = np.radians(
    item_locations["seller_longitude"]
    - item_locations["customer_longitude"])

haversine_term = (
    np.sin((seller_lat - customer_lat) / 2) ** 2
    + np.cos(customer_lat)
    * np.cos(seller_lat)
    * np.sin(longitude_difference / 2) ** 2)

item_locations["distance_km"] = (
    2
    * 6371.0088
    * np.arcsin(np.sqrt(haversine_term.clip(0, 1))))

distance_summary = (
    item_locations.groupby("order_id")
    .agg(
        seller_customer_distance_km_item_mean=("distance_km", "mean"),
        seller_customer_distance_km_max=("distance_km", "max"),
        item_distance_observed_count=("distance_km", "count"),)
    .reset_index())

assert distance_summary["order_id"].is_unique
display(distance_summary.head())

,order_id,seller_customer_distance_km_item_mean,seller_customer_distance_km_max,item_distance_observed_count
0,00010242fe8c5a6d1ba2dd792cb16214,301.299954,301.299954,1
1,00018f77f2f0320c557190d7a144bdd3,588.545069,588.545069,1
2,000229ec398224ef6ca0657da4fc703e,312.955650,312.955650,1
3,00024acbcdf0a6daa1e931b038114c75,295.319989,295.319989,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,646.460714,646.460714,1


In [50]:
orders_merged = checked_left_merge(
    orders,
    customers_enriched,
    on="customer_id",
    name="Orders to customers",
    validate="many_to_one",)

for summary, name in [
    (item_summary, "Order item summaries"),
    (payment_summary, "Order payment summaries"),
    (review_summary, "Order review summaries"),
    (seller_location_summary, "Order seller locations"),
    (distance_summary, "Order distance summaries"),]:
    orders_merged = checked_left_merge(
        orders_merged,
        summary,
        on="order_id",
        name=name,
        validate="one_to_one",)

assert len(orders_merged) == len(orders)
assert orders_merged["order_id"].is_unique
assert set(orders_merged["order_id"]) == set(orders["order_id"])

# Confirm that joins have not changed the monetary totals.
for merged_column, source_df, source_column in [
    ("items_price_total", items, "price"),
    ("freight_value_total", items, "freight_value"),
    ("payment_value_total", payments, "payment_value"),]:
    assert np.isclose(
        orders_merged[merged_column].sum(),
        source_df[source_column].sum(),
    ), f"Total reconciliation failed: {merged_column}"

print("Merged orders:", len(orders_merged))
print("Unique order IDs:", orders_merged["order_id"].nunique())

display(orders_merged.head())

Merged orders: 99441
Unique order IDs: 99441


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_latitude,customer_longitude,customer_geo_unique_row_count,item_count,product_count,seller_count,category_count,items_price_total,freight_value_total,primary_product_category,product_categories,product_categories_english,product_ids,seller_ids,shipping_limit_earliest,shipping_limit_latest,product_weight_g_mean,product_volume_cm3_mean,item_missing_category_count,item_missing_weight_count,item_missing_volume_count,primary_product_category_english,payment_record_count,payment_value_total,payment_types,payment_installments_max,payment_installments_mean,review_record_count,review_score_min,review_score_max,review_score_mean,review_id_latest,review_score_latest,review_creation_date,review_answer_timestamp,review_scores_conflict,seller_states,seller_cities,seller_zip_code_prefixes,seller_customer_distance_km_item_mean,seller_customer_distance_km_max,item_distance_observed_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP,-23.575816,-46.587303,16.0,1.0,1.0,1.0,1.0,29.99,8.72,utilidades_domesticas,utilidades_domesticas,housewares,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,2017-10-06 11:07:15,500.0,1976.0,0.0,0.0,0.0,housewares,3.0,38.71,credit_card|voucher,1.0,1.0,1.0,4.0,4.0,4.0,a54f0611adc9ed256b57ede6b6eb5114,4.0,2017-10-11,2017-10-12 03:43:48,False,SP,maua,09350,18.676602,18.676602,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,-12.126533,-45.008244,14.0,1.0,1.0,1.0,1.0,118.70,22.76,perfumaria,perfumaria,perfumery,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,2018-07-30 03:24:27,400.0,4693.0,0.0,0.0,0.0,perfumery,1.0,141.46,boleto,1.0,1.0,1.0,4.0,4.0,4.0,8d5266042046a06655c8db133d120ba5,4.0,2018-08-08,2018-08-08 18:37:50,False,SP,belo horizonte,31570,861.069892,861.069892,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,-16.744472,-48.514624,25.0,1.0,1.0,1.0,1.0,159.90,19.22,automotivo,automotivo,auto,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,2018-08-13 08:55:23,420.0,9576.0,0.0,0.0,0.0,auto,1.0,179.12,credit_card,3.0,3.0,1.0,5.0,5.0,5.0,e73b67b67587f7644d5bd1a52deb1b01,5.0,2018-08-18,2018-08-22 19:07:58,False,SP,guariba,14840,514.535809,514.535809,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,-5.773971,-35.273838,17.0,1.0,1.0,1.0,1.0,45.00,27.20,pet_shop,pet_shop,pet_shop,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,2017-11-23 19:45:59,450.0,6000.0,0.0,0.0,0.0,pet_shop,1.0,72.20,credit_card,1.0,1.0,1.0,5.0,5.0,5.0,359d03e676b3c069f62cadba8dd3f6e8,5.0,2017-12-03,2017-12-05 19:21:58,False,MG,belo horizonte,31842,1821.874151,1821.874151,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,09195,santo andre,SP,-23.675471,-46.514788,153.0,1.0,1.0,1.0,1.0,19.90,8.72,papelaria,papelaria,stationery,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-0

In [52]:
merge_check_summary = pd.DataFrame(merge_checks)
display(merge_check_summary)

merge_check_summary.to_csv(
    TABLE_DIR / "merge_checks.csv",
    index=False,)

,join,left_rows,output_rows,unmatched_left_rows
0,Customer ZIP to coordinates,99441,99441,278
1,Customer ZIP to coordinates,99441,99441,278
2,Seller ZIP to coordinates,3095,3095,7
3,Items to seller information,112650,112650,0
4,Seller ZIP to coordinates,3095,3095,7
5,Items to seller information,112650,112650,0
6,Item locations to order customer,112650,112650,0
7,Item locations to customer coordinates,112650,112650,0
8,Item locations to order customer,112650,112650,0
9,Item locations to customer coordinates,112650,112650,0


The merged table preserves all 99,441 original orders, with one row per order. Item-price, freight, and payment totals reconcile with
the source records.

Missing customer or seller coordinates remain missing. ZIP-prefix medians are approximate locations, and globally valid coordinates still require geographic quality checks.

Multiple seller locations are retained. Order-level mean distances use observed item-level distances, so coverage counts must accompany
their interpretation.

## 5. Derived Measurements and Preliminary Eligibility

Durations use recorded timestamps. Delivery delay compares calendar dates, so delivery on the estimated date is classified as on time.

Missing outcomes remain unobserved. Quality flags identify records for later investigation without automatically deleting orders.

In [55]:
def elapsed_days(end_column, start_column):
    """Calculate elapsed days while preserving missing timestamps."""
    return (
        orders_merged[end_column] - orders_merged[start_column]).dt.total_seconds() / 86400


duration_pairs = {
    "approval_duration_days": (
        "order_approved_at",
        "order_purchase_timestamp",
    ),
    "seller_handling_duration_days": (
        "order_delivered_carrier_date",
        "order_approved_at",
    ),
    "carrier_delivery_duration_days": (
        "order_delivered_customer_date",
        "order_delivered_carrier_date",
    ),
    "total_delivery_duration_days": (
        "order_delivered_customer_date",
        "order_purchase_timestamp",
    ),
    "promised_delivery_duration_days": (
        "order_estimated_delivery_date",
        "order_purchase_timestamp",),}

for output_column, (end_column, start_column) in duration_pairs.items():
    orders_merged[output_column] = elapsed_days(
        end_column,
        start_column,)

# Calendar-date comparison: same-day delivery is on time.
orders_merged["delivery_delay_days"] = (
    orders_merged["order_delivered_customer_date"].dt.normalize()
    - orders_merged["order_estimated_delivery_date"].dt.normalize()).dt.days.astype("Int64")

orders_merged["delivered_late"] = (
    orders_merged["delivery_delay_days"].gt(0)).astype("boolean")

positive_item_value = orders_merged["items_price_total"].where(
    orders_merged["items_price_total"].gt(0))

orders_merged["freight_to_items_value_ratio"] = (
    orders_merged["freight_value_total"] / positive_item_value)

orders_merged["order_purchase_month"] = (
    orders_merged["order_purchase_timestamp"].dt.strftime("%Y-%m"))

In [57]:
orders_merged["has_item_records"] = (
    orders_merged["item_count"].notna())

orders_merged["has_payment_records"] = (
    orders_merged["payment_record_count"].notna())

orders_merged["has_review"] = (
    orders_merged["review_record_count"].notna())

valid_selected_score = orders_merged[
    "review_score_latest"].between(1, 5)

orders_merged["has_valid_review_score"] = valid_selected_score

orders_merged["is_delivered"] = (
    orders_merged["order_status"].eq("delivered"))

orders_merged["eligible_delivered_reviewed"] = (
    orders_merged["is_delivered"] & valid_selected_score)

orders_merged["dissatisfied_latest"] = (
    orders_merged["review_score_latest"]
    .le(2)
    .where(valid_selected_score)
    .astype("boolean"))

valid_minimum_score = orders_merged[
    "review_score_min"].between(1, 5)

orders_merged["dissatisfied_any_review"] = (
    orders_merged["review_score_min"]
    .le(2)
    .where(valid_minimum_score)
    .astype("boolean"))

In [59]:
duration_columns = list(duration_pairs)

orders_merged["negative_duration_flag"] = (
    orders_merged[duration_columns].lt(0).any(axis=1))

known_review_and_delivery_times = (
    orders_merged["review_answer_timestamp"].notna()
    & orders_merged["order_delivered_customer_date"].notna())

orders_merged["review_before_delivery"] = (
    orders_merged["review_answer_timestamp"]
    .lt(orders_merged["order_delivered_customer_date"])
    .where(known_review_and_delivery_times)
    .astype("boolean"))

# Missing when an order has no item records.
orders_merged["distance_coverage_ratio"] = (
    orders_merged["item_distance_observed_count"]
    / orders_merged["item_count"])

display(
    orders_merged[
        [
            "order_id",
            "total_delivery_duration_days",
            "delivery_delay_days",
            "review_score_latest",
            "dissatisfied_latest",
            "eligible_delivered_reviewed",
            "review_before_delivery",
        ]].head())

,order_id,total_delivery_duration_days,delivery_delay_days,review_score_latest,dissatisfied_latest,eligible_delivered_reviewed,review_before_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,-8,4.0,False,True,False
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,-6,4.0,False,True,False
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,-18,5.0,False,True,False
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,-13,5.0,False,True,False
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,-10,5.0,False,True,False


In [60]:
assert len(orders_merged) == len(orders)
assert orders_merged["order_id"].notna().all()
assert orders_merged["order_id"].is_unique
assert set(orders_merged["order_id"]) == set(orders["order_id"])

# Absent valid reviews must have absent outcome labels.
assert orders_merged.loc[
    ~orders_merged["has_valid_review_score"],
    "dissatisfied_latest",
].isna().all()

# Absent delivery dates must not be classified as on time.
assert orders_merged.loc[
    orders_merged["delivery_delay_days"].isna(),
    "delivered_late",
].isna().all()

eligible_mask = orders_merged["eligible_delivered_reviewed"]

merge_overview = pd.DataFrame([
    {
        "metric": "Total orders",
        "value": len(orders_merged),
    },
    {
        "metric": "Unique order IDs",
        "value": orders_merged["order_id"].nunique(),
    },
    {
        "metric": "Delivered orders",
        "value": int(orders_merged["is_delivered"].sum()),
    },
    {
        "metric": "Orders with review records",
        "value": int(orders_merged["has_review"].sum()),
    },
    {
        "metric": "Preliminary delivered-and-reviewed orders",
        "value": int(eligible_mask.sum()),
    },
    {
        "metric": "Low reviews in preliminary cohort",
        "value": int(
            orders_merged.loc[
                eligible_mask, "dissatisfied_latest"
            ].sum()
        ),
    },
    {
        "metric": "Orders flagged for negative durations",
        "value": int(orders_merged["negative_duration_flag"].sum()),
    },
    {
        "metric": "Orders reviewed before recorded delivery",
        "value": int(orders_merged["review_before_delivery"].sum()),
    },])

display(merge_overview)

,metric,value
0,Total orders,99441
1,Unique order IDs,99441
2,Delivered orders,96478
3,Orders with review records,98673
4,Preliminary delivered-and-reviewed orders,95832
5,Low reviews in preliminary cohort,12273
6,Orders flagged for negative durations,1382
7,Orders reviewed before recorded delivery,4654


## 6. Export and Reproducibility Checks

The merged dataset is exported for the separate analysis notebook. CSV files do not preserve pandas data types, so timestamp, identifier,
and nullable Boolean columns must be restored when loading them.
Source file hashes identify the exact input files used in this run.

In [63]:
import hashlib

MERGED_PATH = DERIVED_DIR / "olist_orders_merged.csv"

orders_merged.to_csv(MERGED_PATH, index=False)

merge_overview.to_csv(
    TABLE_DIR / "merge_overview.csv",
    index=False,)

column_inventory = pd.DataFrame({
    "column": orders_merged.columns,
    "pandas_dtype": [
        str(orders_merged[column].dtype)
        for column in orders_merged.columns
    ],
    "missing_count": [
        int(orders_merged[column].isna().sum())
        for column in orders_merged.columns
    ],})

column_inventory.to_csv(
    DERIVED_DIR / "merged_column_inventory.csv",
    index=False,)


def sha256_file(path):
    """Hash in chunks without loading the entire file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

source_manifest = pd.DataFrame([
    {
        "table": name,
        "filename": filename,
        "rows": len(raw_tables[name]),
        "sha256": sha256_file(RAW_DIR / filename),
    }
    for name, filename in SOURCE_FILES.items()])

source_manifest.to_csv(
    DERIVED_DIR / "source_manifest.csv",
    index=False,)

print("Saved merged dataset:", MERGED_PATH)
print("Rows:", len(orders_merged))
print("Columns:", orders_merged.shape[1])

Saved merged dataset: /Users/bishanzhou/Documents/wustl_life/practicum/olist_capstone/data/derived/olist_orders_merged.csv
Rows: 99441
Columns: 74


In [65]:
export_check = pd.read_csv(
    MERGED_PATH,
    usecols=["order_id"],
    dtype={"order_id": "string"},)

assert len(export_check) == len(orders_merged)
assert export_check["order_id"].notna().all()
assert export_check["order_id"].is_unique
assert set(export_check["order_id"]) == set(orders["order_id"])

print("Export checks passed.")

Export checks passed.


## Preparation Summary

The workflow preserves all 99,441 orders with one row per order. Item-price, freight, and payment totals reconcile with their sources.

Derived durations, delay measures, review labels, and diagnostic flags are available for analysis. The preliminary delivered-and-reviewed
cohort contains 95,832 orders.

No automatic exclusion or imputation has been applied to flagged measurements. Final analytical eligibility and sensitivity analyses
are addressed in 02_milestone2_analysis.ipynb.

The exported dataset is data/derived/olist_orders_merged.csv.